In [ ]:
"""
Comprehensive ML Models for India EXIM Critical Minerals Forecasting
Models: ARIMA, LSTM, Hybrid ARIMA-LSTM
Minerals: Copper, Lithium, Graphite
"""

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Statistical and ML Libraries
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

============================================================================
SECTION 1: DATA PREPROCESSING
============================================================================

In [ ]:
class EXIMDataProcessor:
    """Preprocesses DGCI&S EXIM data for time series analysis"""
    
    def __init__(self, file_path=None):
        self.file_path = file_path
        self.data = None
        self.scaler = MinMaxScaler(feature_range=(0, 1))
        
    def load_data(self, sample_data=True):
        """Load EXIM data from CSV or create sample data"""
        if sample_data:
            # Generate sample data for demonstration
            dates = pd.date_range(start='2017-04-01', end='2024-03-01', freq='M')
            
            # Realistic patterns with trend and seasonality
            n = len(dates)
            trend = np.linspace(40000, 75000, n)
            seasonal = 5000 * np.sin(np.linspace(0, 6*np.pi, n))
            noise = np.random.normal(0, 2000, n)
            
            copper_import = trend + seasonal + noise
            copper_export = copper_import * 0.18 + np.random.normal(0, 500, n)
            
            lithium_import = np.linspace(8000, 32000, n) + 2000 * np.sin(np.linspace(0, 6*np.pi, n)) + np.random.normal(0, 1000, n)
            lithium_export = lithium_import * 0.015 + np.random.normal(0, 100, n)
            
            graphite_import = np.linspace(15000, 28000, n) + 1500 * np.sin(np.linspace(0, 6*np.pi, n)) + np.random.normal(0, 800, n)
            graphite_export = graphite_import * 0.35 + np.random.normal(0, 400, n)
            
            self.data = pd.DataFrame({
                'date': dates,
                'copper_import': copper_import,
                'copper_export': copper_export,
                'lithium_import': lithium_import,
                'lithium_export': lithium_export,
                'graphite_import': graphite_import,
                'graphite_export': graphite_export
            })
            
            self.data['year'] = self.data['date'].dt.year
            self.data['month'] = self.data['date'].dt.month
            self.data['quarter'] = self.data['date'].dt.quarter
            
        else:
            # Load from CSV file (DGCI&S format)
            self.data = pd.read_csv(self.file_path)
            self.data['date'] = pd.to_datetime(self.data['date'])
            
        return self.data
    
    def check_stationarity(self, series, series_name):
        """Perform Augmented Dickey-Fuller test for stationarity"""
        clean_series = pd.Series(series).dropna()
        result = adfuller(clean_series)
        
        print(f'\n=== Stationarity Test: {series_name} ===')
        print(f'ADF Statistic: {result[0]:.4f}')
        print(f'p-value: {result[1]:.4f}')
        print(f'Critical Values:')
        for key, value in result[4].items():
            print(f'\t{key}: {value:.3f}')
        
        if result[1] <= 0.05:
            print(f"✓ Series is stationary (reject null hypothesis)")
            return True
        else:
            print(f"✗ Series is non-stationary (fail to reject null hypothesis)")
            return False
    
    def difference_series(self, series, order=1):
        """Apply differencing to make series stationary"""
        return series.diff(order).dropna()
    
    def create_sequences(self, data, n_steps):
        """Create sequences for LSTM model"""
        X, y = [], []
        for i in range(len(data) - n_steps):
            X.append(data[i:i+n_steps])
            y.append(data[i+n_steps])
        return np.array(X), np.array(y)
    
    def split_data(self, series, train_ratio=0.8):
        """Split data into train and test sets"""
        split_idx = int(len(series) * train_ratio)
        train = series[:split_idx]
        test = series[split_idx:]
        return train, test

============================================================================
SECTION 2: ARIMA MODEL
============================================================================

In [ ]:
class ARIMAForecaster:
    """ARIMA model for time series forecasting"""
    
    def __init__(self, order=(1, 1, 1)):
        self.order = order
        self.model = None
        self.fitted_model = None
        
    def find_optimal_order(self, series, max_p=5, max_d=2, max_q=5):
        """Find optimal ARIMA order using AIC/BIC"""
        best_aic = np.inf
        best_order = None
        
        print("\n=== Finding Optimal ARIMA Order ===")
        
        for p in range(max_p + 1):
            for d in range(max_d + 1):
                for q in range(max_q + 1):
                    try:
                        model = ARIMA(series, order=(p, d, q))
                        fitted = model.fit()
                        
                        if fitted.aic < best_aic:
                            best_aic = fitted.aic
                            best_order = (p, d, q)
                            
                    except:
                        continue
        
        print(f"Best ARIMA order: {best_order}")
        print(f"Best AIC: {best_aic:.2f}")
        
        self.order = best_order
        return best_order
    
    def fit(self, train_data):
        """Fit ARIMA model"""
        print(f"\n=== Fitting ARIMA{self.order} ===")
        self.model = ARIMA(train_data, order=self.order)
        self.fitted_model = self.model.fit()
        
        print(self.fitted_model.summary())
        return self.fitted_model
    
    def forecast(self, steps=12):
        """Generate forecasts"""
        forecast = self.fitted_model.forecast(steps=steps)
        
        # Get confidence intervals
        forecast_df = self.fitted_model.get_forecast(steps=steps).summary_frame()
        
        return forecast, forecast_df
    
    def plot_diagnostics(self):
        """Plot model diagnostics"""
        fig, axes = plt.subplots(2, 2, figsize=(14, 8))
        
        # Residuals plot
        residuals = self.fitted_model.resid
        axes[0, 0].plot(residuals)
        axes[0, 0].set_title('Residuals')
        axes[0, 0].set_xlabel('Time')
        
        # Histogram of residuals
        axes[0, 1].hist(residuals, bins=30, edgecolor='black')
        axes[0, 1].set_title('Histogram of Residuals')
        
        # ACF of residuals
        plot_acf(residuals, lags=20, ax=axes[1, 0])
        axes[1, 0].set_title('ACF of Residuals')
        
        # Q-Q plot
        from scipy import stats
        stats.probplot(residuals, dist="norm", plot=axes[1, 1])
        axes[1, 1].set_title('Q-Q Plot')
        
        plt.tight_layout()
        return fig

============================================================================
SECTION 3: LSTM MODEL
============================================================================

In [ ]:
class LSTMForecaster:
    """LSTM neural network for time series forecasting"""
    
    def __init__(self, n_steps=12, n_features=1):
        self.n_steps = n_steps
        self.n_features = n_features
        self.model = None
        self.scaler = MinMaxScaler(feature_range=(0, 1))
        self.history = None
        
    def build_model(self, units=[50, 50], dropout=0.2):
        """Build LSTM architecture"""
        self.model = Sequential()
        
        # First LSTM layer
        self.model.add(LSTM(units=units[0], return_sequences=True, 
                           input_shape=(self.n_steps, self.n_features)))
        self.model.add(Dropout(dropout))
        
        # Second LSTM layer
        self.model.add(LSTM(units=units[1], return_sequences=False))
        self.model.add(Dropout(dropout))
        
        # Output layer
        self.model.add(Dense(units=1))
        
        # Compile model
        self.model.compile(optimizer='adam', loss='mean_squared_error', 
                          metrics=['mae'])
        
        print("\n=== LSTM Model Architecture ===")
        self.model.summary()
        
        return self.model
    
    def prepare_data(self, data):
        """Scale and prepare data for LSTM"""
        # Ensure data is a numpy array before reshaping
        data_array = np.array(data).reshape(-1, 1)
        scaled_data = self.scaler.fit_transform(data_array)
        return scaled_data
    
    def fit(self, train_data, epochs=100, batch_size=32, validation_split=0.1):
        """Train LSTM model"""
        # Prepare data
        scaled_train = self.prepare_data(train_data)
        X_train, y_train = self.create_sequences(scaled_train)
        
        # Early stopping
        early_stop = EarlyStopping(monitor='val_loss', patience=10, 
                                   restore_best_weights=True)
        
        # Train model
        print("\n=== Training LSTM Model ===")
        self.history = self.model.fit(
            X_train, y_train,
            epochs=epochs,
            batch_size=batch_size,
            validation_split=validation_split,
            callbacks=[early_stop],
            verbose=1
        )
        
        return self.history
    
    def create_sequences(self, data):
        """Create input sequences for LSTM"""
        X, y = [], []
        for i in range(len(data) - self.n_steps):
            X.append(data[i:i+self.n_steps, 0])
            y.append(data[i+self.n_steps, 0])
        return np.array(X).reshape(-1, self.n_steps, 1), np.array(y)
    
    def forecast(self, last_sequence, steps=12):
        """Generate multi-step forecasts"""
        forecasts = []
        current_sequence = last_sequence.copy()
        
        for _ in range(steps):
            # Reshape for prediction
            input_seq = current_sequence.reshape(1, self.n_steps, 1)
            
            # Predict next value
            pred = self.model.predict(input_seq, verbose=0)
            forecasts.append(pred[0, 0])
            
            # Update sequence
            current_sequence = np.append(current_sequence[1:], pred)
        
        # Inverse transform
        forecasts = np.array(forecasts).reshape(-1, 1)
        forecasts = self.scaler.inverse_transform(forecasts)
        
        return forecasts.flatten()
    
    def plot_training_history(self):
        """Plot training and validation loss"""
        fig, ax = plt.subplots(1, 2, figsize=(14, 4))
        
        ax[0].plot(self.history.history['loss'], label='Training Loss')
        ax[0].plot(self.history.history['val_loss'], label='Validation Loss')
        ax[0].set_xlabel('Epoch')
        ax[0].set_ylabel('Loss')
        ax[0].set_title('Model Loss')
        ax[0].legend()
        ax[0].grid(True)
        
        ax[1].plot(self.history.history['mae'], label='Training MAE')
        ax[1].plot(self.history.history['val_mae'], label='Validation MAE')
        ax[1].set_xlabel('Epoch')
        ax[1].set_ylabel('MAE')
        ax[1].set_title('Model MAE')
        ax[1].legend()
        ax[1].grid(True)
        
        plt.tight_layout()
        return fig

============================================================================
SECTION 4: HYBRID ARIMA-LSTM MODEL
============================================================================

In [ ]:
class HybridARIMALSTM:
    """Hybrid model combining ARIMA and LSTM"""
    
    def __init__(self, arima_order=(1, 1, 1), lstm_steps=12):
        self.arima_model = ARIMAForecaster(order=arima_order)
        self.lstm_model = LSTMForecaster(n_steps=lstm_steps)
        self.arima_weight = 0.5
        self.lstm_weight = 0.5
        
    def fit(self, train_data, optimize_arima=True, epochs=100):
        """Fit both ARIMA and LSTM models"""
        print("\n" + "="*70)
        print("HYBRID ARIMA-LSTM MODEL TRAINING")
        print("="*70)
        
        # Fit ARIMA
        if optimize_arima:
            self.arima_model.find_optimal_order(train_data)
        
        self.arima_model.fit(train_data)
        
        # Fit LSTM
        self.lstm_model.build_model()
        self.lstm_model.fit(train_data, epochs=epochs)
        
        # Optimize weights based on validation performance
        self.optimize_weights(train_data)
        
    def optimize_weights(self, data, validation_size=12):
        """Optimize ensemble weights"""
        # Split data for validation
        val_data = data[-validation_size:]
        train_for_val = data[:-validation_size]
        
        best_mae = np.inf
        best_weights = (0.5, 0.5)
        
        for w1 in np.arange(0, 1.1, 0.1):
            w2 = 1 - w1
            
            # Get predictions from both models
            arima_pred, _ = self.arima_model.forecast(steps=validation_size)
            
            last_seq = self.lstm_model.prepare_data(train_for_val)[-self.lstm_model.n_steps:]
            lstm_pred = self.lstm_model.forecast(last_seq, steps=validation_size)
            
            # Ensemble prediction
            ensemble_pred = w1 * arima_pred + w2 * lstm_pred
            
            # Calculate MAE
            mae = mean_absolute_error(val_data, ensemble_pred)
            
            if mae < best_mae:
                best_mae = mae
                best_weights = (w1, w2)
        
        self.arima_weight, self.lstm_weight = best_weights
        print(f"\n=== Optimized Weights ===")
        print(f"ARIMA: {self.arima_weight:.2f}, LSTM: {self.lstm_weight:.2f}")
        print(f"Validation MAE: {best_mae:.2f}")
    
    def forecast(self, last_data, steps=12):
        """Generate hybrid forecasts"""
        # ARIMA forecast
        arima_forecast, arima_ci = self.arima_model.forecast(steps=steps)
        
        # LSTM forecast
        last_seq = self.lstm_model.prepare_data(last_data)[-self.lstm_model.n_steps:]
        lstm_forecast = self.lstm_model.forecast(last_seq, steps=steps)
        
        # Ensemble forecast
        hybrid_forecast = (self.arima_weight * arima_forecast + 
                          self.lstm_weight * lstm_forecast)
        
        # Calculate confidence intervals (approximate)
        std_error = np.std([arima_forecast, lstm_forecast], axis=0)
        lower_ci = hybrid_forecast - 1.96 * std_error
        upper_ci = hybrid_forecast + 1.96 * std_error
        
        forecast_df = pd.DataFrame({
            'forecast': hybrid_forecast,
            'lower_ci': lower_ci,
            'upper_ci': upper_ci,
            'arima_component': arima_forecast,
            'lstm_component': lstm_forecast
        })
        
        return forecast_df

============================================================================
SECTION 5: MODEL EVALUATION
============================================================================

In [ ]:
class ModelEvaluator:
    """Evaluate and compare forecasting models"""
    
    @staticmethod
    def calculate_metrics(actual, predicted):
        """Calculate comprehensive evaluation metrics"""
        mae = mean_absolute_error(actual, predicted)
        rmse = np.sqrt(mean_squared_error(actual, predicted))
        mape = np.mean(np.abs((actual - predicted) / actual)) * 100
        r2 = r2_score(actual, predicted)
        
        return {
            'MAE': mae,
            'RMSE': rmse,
            'MAPE': mape,
            'R2': r2
        }
    
    @staticmethod
    def compare_models(actual, predictions_dict):
        """Compare multiple models"""
        results = []
        
        for model_name, predictions in predictions_dict.items():
            metrics = ModelEvaluator.calculate_metrics(actual, predictions)
            metrics['Model'] = model_name
            results.append(metrics)
        
        results_df = pd.DataFrame(results)
        results_df = results_df[['Model', 'MAE', 'RMSE', 'MAPE', 'R2']]
        
        return results_df
    
    @staticmethod
    def plot_forecast_comparison(actual, predictions_dict, title="Model Comparison"):
        """Plot actual vs predicted for multiple models"""
        fig, ax = plt.subplots(figsize=(14, 6))
        
        # Plot actual values
        ax.plot(range(len(actual)), actual, label='Actual', 
               linewidth=2, color='black', marker='o')
        
        # Plot predictions
        colors = ['blue', 'red', 'green', 'orange', 'purple']
        for idx, (model_name, predictions) in enumerate(predictions_dict.items()):
            ax.plot(range(len(predictions)), predictions, 
                   label=model_name, linewidth=2, 
                   color=colors[idx % len(colors)], 
                   marker='s', alpha=0.7)
        
        ax.set_xlabel('Time Period', fontsize=12)
        ax.set_ylabel('Value (₹ Crores)', fontsize=12)
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        return fig

============================================================================
SECTION 6: MAIN EXECUTION PIPELINE
============================================================================

In [ ]:
def main():
    """Main execution pipeline for EXIM forecasting"""
    
    print("="*70)
    print("INDIA EXIM CRITICAL MINERALS FORECASTING SYSTEM")
    print("="*70)
    
    # 1. Load and preprocess data
    processor = EXIMDataProcessor()
    data = processor.load_data(sample_data=True)
    
    print(f"\nData loaded: {len(data)} observations")
    print(f"Date range: {data['date'].min()} to {data['date'].max()}")
    
    # 2. Focus on Copper imports for demonstration
    series = data['copper_import']
    
    # Check stationarity
    processor.check_stationarity(series, "Copper Imports")
    
    # Split data
    train, test = processor.split_data(series, train_ratio=0.85)
    print(f"\nTrain size: {len(train)}, Test size: {len(test)}")
    
    # 3. Train ARIMA model
    arima_model = ARIMAForecaster()
    arima_model.find_optimal_order(train)
    arima_model.fit(train)
    arima_forecast, _ = arima_model.forecast(steps=len(test))
    
    # 4. Train LSTM model
    lstm_model = LSTMForecaster(n_steps=12)
    lstm_model.build_model(units=[64, 32])
    lstm_model.fit(train, epochs=50, batch_size=16)
    
    last_seq = lstm_model.prepare_data(train)[-12:]
    lstm_forecast = lstm_model.forecast(last_seq, steps=len(test))
    
    # 5. Train Hybrid model
    hybrid_model = HybridARIMALSTM(lstm_steps=12)
    hybrid_model.fit(train, optimize_arima=False, epochs=50)
    hybrid_forecast_df = hybrid_model.forecast(train, steps=len(test))
    
    # 6. Evaluate models
    evaluator = ModelEvaluator()
    
    predictions = {
        'ARIMA': arima_forecast,
        'LSTM': lstm_forecast,
        'Hybrid': hybrid_forecast_df['forecast'].values
    }
    
    metrics_df = evaluator.compare_models(test, predictions)
    
    print("\n" + "="*70)
    print("MODEL PERFORMANCE COMPARISON")
    print("="*70)
    print(metrics_df.to_string(index=False))
    
    # 7. Generate future forecasts
    print("\n" + "="*70)
    print("GENERATING 12-MONTH AHEAD FORECASTS")
    print("="*70)
    
    future_forecast = hybrid_model.forecast(series, steps=12)
    print("\nFuture Forecast:")
    print(future_forecast)
    
    # 8. Visualizations
    fig = evaluator.plot_forecast_comparison(
        test, predictions, 
        title="Copper Import Forecasts: Model Comparison"
    )
    plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
    print("\n✓ Model comparison plot saved as 'model_comparison.png'")
    
    lstm_model.plot_training_history()
    plt.savefig('lstm_training.png', dpi=300, bbox_inches='tight')
    print("✓ LSTM training history saved as 'lstm_training.png'")
    
    arima_model.plot_diagnostics()
    plt.savefig('arima_diagnostics.png', dpi=300, bbox_inches='tight')
    print("✓ ARIMA diagnostics saved as 'arima_diagnostics.png'")
    
    print("\n" + "="*70)
    print("FORECASTING PIPELINE COMPLETED SUCCESSFULLY")
    print("="*70)

In [ ]:
if __name__ == "__main__":
    main()